# eICU to JSON Asplenia Workflow with ICD-Code-Based Cohort Selection and Synthetic Dates

This notebook implements an eICU-CRD equivalent of the MIMIC-IV asplenia workflow. It selects the cohort by comparing `diagnosis.icd9code` with external ICD code lists and converts eICU offsets into synthetic dates for compatibility with algorithms that require date fields.

Important: eICU does not provide real calendar dates. The generated `date` field is synthetic and preserves only relative temporal order.


In [1]:
import os
import re
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

## 1. Configuration

In [2]:
DATA_PATH = Path("../physionet.org/files/eicu-crd/2.0/")

PATIENT_FILE = DATA_PATH / "patient.csv"
DIAGNOSIS_FILE = DATA_PATH / "diagnosis.csv"
TREATMENT_FILE = DATA_PATH / "treatment.csv"
MEDICATION_FILE = DATA_PATH / "medication.csv"

ICD_CODE_LIST_DIR = Path("icd9_code_lists")
ICD_CODE_FILE_EXTENSIONS = [".txt", ".csv"]
USE_TEXT_FALLBACK_FOR_COHORT = False

# Synthetic reference date. This is not a real calendar date.
SYNTHETIC_REFERENCE_DATE = pd.Timestamp("2100-01-01 00:00:00")

OUTPUT_JSON = "eicu_asplenia_patients_synthetic_dates.json"
OUTPUT_EVENTS_CSV = "eicu_asplenia_events_synthetic_dates.csv"
OUTPUT_COHORT_CSV = "eicu_asplenia_cohort.csv"

## 2. Utility Functions

In [3]:
def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path, low_memory=False, **kwargs)


def match_patterns(text, patterns):
    if pd.isna(text):
        return False
    text = str(text).lower()
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in patterns)


def normalize_icd_code(code):
    if pd.isna(code):
        return None
    code = str(code).strip().upper()
    code = re.sub(r"[^A-Z0-9]", "", code)
    return code if code else None


def split_eicu_icd_codes(value):
    if pd.isna(value):
        return []
    parts = re.split(r"[,;| ]+", str(value))
    return [normalize_icd_code(p) for p in parts if normalize_icd_code(p)]


def load_icd_code_lists(code_dir):
    code_dir = Path(code_dir)
    code_to_category = {}
    category_to_codes = {}
    if not code_dir.exists():
        print(f"ICD code list directory not found: {code_dir}")
        print("No ICD-code-based patients will be selected unless text fallback is enabled.")
        return code_to_category, category_to_codes
    files = [p for p in code_dir.iterdir() if p.is_file() and p.suffix.lower() in ICD_CODE_FILE_EXTENSIONS]
    for file in files:
        category = file.stem
        codes = set()
        if file.suffix.lower() == ".txt":
            for line in file.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                code = normalize_icd_code(line)
                if code:
                    codes.add(code)
        elif file.suffix.lower() == ".csv":
            df = pd.read_csv(file, dtype=str)
            preferred = [c for c in df.columns if c.lower() in ["code", "icd", "icd9", "icd9code", "icd_code"]]
            col = preferred[0] if preferred else df.columns[0]
            for value in df[col].dropna():
                code = normalize_icd_code(value)
                if code:
                    codes.add(code)
        category_to_codes[category] = codes
        for code in codes:
            code_to_category[code] = category
    print(f"Loaded {sum(len(v) for v in category_to_codes.values())} ICD codes from {len(category_to_codes)} categories.")
    return code_to_category, category_to_codes


def assign_icd_category(icd_value, code_to_category):
    for code in split_eicu_icd_codes(icd_value):
        if code in code_to_category:
            return code_to_category[code]
    return None


def normalize_age(age_value):
    if pd.isna(age_value):
        return None
    value = str(age_value).strip()
    if value.startswith(">"):
        digits = re.findall(r"\d+", value)
        return int(digits[0]) if digits else None
    digits = re.findall(r"\d+", value)
    return int(digits[0]) if digits else None


def age_group(age):
    if age is None or pd.isna(age):
        return "unknown"
    if age < 40:
        return "young"
    if age < 65:
        return "mature"
    if age < 80:
        return "elder"
    return "geriatric"


def offset_to_synthetic_datetime(offset, reference_date=SYNTHETIC_REFERENCE_DATE):
    if pd.isna(offset):
        return pd.NaT
    return reference_date + pd.to_timedelta(float(offset), unit="m")


def clean_value(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


def clean_datetime(value):
    if value is None or pd.isna(value):
        return None
    return pd.Timestamp(value).isoformat()


def as_bool_alive(status):
    if pd.isna(status):
        return None
    return str(status).strip().lower() != "expired"

## 3. Load eICU Tables

In [4]:
patient = read_csv_safe(PATIENT_FILE)
diagnosis = read_csv_safe(DIAGNOSIS_FILE)
treatment = read_csv_safe(TREATMENT_FILE)
medication = read_csv_safe(MEDICATION_FILE)

print("patient:", patient.shape)
print("diagnosis:", diagnosis.shape)
print("treatment:", treatment.shape)
print("medication:", medication.shape)

patient: (200859, 29)
diagnosis: (2710672, 7)
treatment: (3688745, 5)
medication: (7301853, 15)


## 4. Clinical Text Patterns

In [5]:
asplenia_text_patterns = [
    r"\basplenia\b", r"absence of spleen", r"absent spleen", r"postsplenectomy",
    r"post[- ]?splenectomy", r"\bsplenectomy\b", r"splenic dysfunction", r"functional asplenia",
    r"sickle cell", r"sickle-cell", r"thalassemia", r"thalassaemia", r"hemolytic anemia", r"haemolytic anaemia"
]

infection_patterns = [
    r"\bsepsis\b", r"septicemia", r"septic shock", r"pneumonia", r"meningitis", r"encephalitis",
    r"bacteremia", r"bacteraemia", r"endocarditis", r"cellulitis", r"\babscess\b",
    r"urinary tract infection", r"\buti\b", r"fungal infection", r"viral infection", r"tuberculosis",
    r"peritonitis", r"wound infection"
]

vaccine_patterns = [
    r"vaccine", r"vaccination", r"immunization", r"immunisation", r"pneumococ", r"meningococ",
    r"influenza", r"haemophilus", r"haemophilus influenzae", r"haemophilus influenzae type b",
    r"\bhib\b", r"hepatitis", r"tdap", r"tetanus", r"diphtheria", r"pertussis", r"covid", r"sars[- ]?cov[- ]?2"
]

splenectomy_patterns = [
    r"\bsplenectomy\b", r"post[- ]?splenectomy", r"spleen removal", r"removal of spleen", r"excision.*spleen", r"resection.*spleen"
]

## 5. ICD-Code-Based Cohort Selection

In [6]:
code_to_category, category_to_codes = load_icd_code_lists(ICD_CODE_LIST_DIR)

diagnosis = diagnosis.copy()
diagnosis["diagnosisstring_clean"] = diagnosis["diagnosisstring"].astype(str).str.lower()
diagnosis["icd9code_clean"] = diagnosis["icd9code"].astype(str).str.lower()

diagnosis["icd_category"] = diagnosis["icd9code"].apply(lambda x: assign_icd_category(x, code_to_category))
diagnosis["icd_code_match"] = diagnosis["icd_category"].notna()

if USE_TEXT_FALLBACK_FOR_COHORT:
    diagnosis["text_asplenia_match"] = diagnosis["diagnosisstring"].apply(lambda x: match_patterns(x, asplenia_text_patterns))
else:
    diagnosis["text_asplenia_match"] = False

diagnosis["asplenia_related"] = diagnosis["icd_code_match"] | diagnosis["text_asplenia_match"]

asplenic_ids = sorted(diagnosis.loc[diagnosis["asplenia_related"], "patientunitstayid"].dropna().unique())
patient_cohort = patient[patient["patientunitstayid"].isin(asplenic_ids)].copy()

print("Number of asplenia-related ICU stays:", len(asplenic_ids))
print("Cohort patient table shape:", patient_cohort.shape)

diagnosis.loc[diagnosis["asplenia_related"], ["patientunitstayid", "diagnosisstring", "icd9code", "icd_category", "icd_code_match", "text_asplenia_match"]].head(20)

Loaded 66 ICD codes from 5 categories.
Number of asplenia-related ICU stays: 444
Cohort patient table shape: (444, 29)


,patientunitstayid,diagnosisstring,icd9code,icd_category,icd_code_match,text_asplenia_match
10458,156843,gastrointestinal|trauma|splenic trauma,"865.00, S36.0",Onco,True,False
10479,156843,gastrointestinal|trauma|splenic trauma,"865.00, S36.0",Onco,True,False
10501,156843,gastrointestinal|trauma|splenic trauma,"865.00, S36.0",Onco,True,False
10518,156843,gastrointestinal|trauma|splenic trauma,"865.00, S36.0",Onco,True,False
29399,182613,burns/trauma|trauma - abdomen|splenic trauma,"865.00, S36.0",Onco,True,False
44955,205175,burns/trauma|trauma - abdomen|splenic trauma|grade IV laceration,865.00,Onco,True,False
56429,219060,burns/trauma|trauma - abdomen|splenic trauma,"865.00, S36.0",Onco,True,False
58783,222365,burns/trauma|trauma - abdomen|splenic trauma,"865.00, S36.0",Onco,True,False
77565,247722,burns/trauma|trauma - abdomen|splenic trauma|grade II laceration,865.00,Onco,True,False
77566,247722,burns/trauma|trauma - abdomen|splenic trauma|grade II laceration,865.00,Onco,True,False


## 6. Add Demographic, Disease Category, and Outcome Features

In [7]:
patient_cohort["age_numeric"] = patient_cohort["age"].apply(normalize_age)
patient_cohort["age_group"] = patient_cohort["age_numeric"].apply(age_group)
patient_cohort["alive"] = patient_cohort["hospitaldischargestatus"].apply(as_bool_alive)

primary_category = (
    diagnosis.loc[diagnosis["icd_category"].notna(), ["patientunitstayid", "icd_category"]]
    .drop_duplicates()
    .groupby("patientunitstayid")["icd_category"]
    .first()
)
patient_cohort["primary_disease_category"] = patient_cohort["patientunitstayid"].map(primary_category)

static_columns = [
    "patientunitstayid", "gender", "age", "age_numeric", "age_group", "ethnicity",
    "hospitalid", "wardid", "apacheadmissiondx", "primary_disease_category",
    "admissionheight", "admissionweight", "hospitaladmitoffset", "hospitaldischargeoffset",
    "unitadmitoffset", "unitdischargeoffset", "unitdischargestatus", "hospitaldischargestatus", "alive"
]
existing_static_columns = [c for c in static_columns if c in patient_cohort.columns]
patient_static = patient_cohort[existing_static_columns].copy()
patient_static.head()

,patientunitstayid,gender,age,age_numeric,age_group,ethnicity,hospitalid,wardid,apacheadmissiondx,primary_disease_category,admissionheight,admissionweight,hospitaladmitoffset,hospitaldischargeoffset,unitdischargeoffset,unitdischargestatus,hospitaldischargestatus,alive
2348,156843,Female,31,31,young,Asian,73,97,"Sepsis, other",Onco,157.5,75.8,-338,57678,57410,Expired,Expired,False
6191,182613,Female,66,66,elder,Caucasian,71,87,Abdomen/multiple trauma,Onco,157.5,NaN,-18,5229,2872,Alive,Alive,True
9381,205175,Male,21,21,young,Caucasian,67,109,Abdomen only trauma,Onco,182.9,67.3,-87,6782,1003,Alive,Alive,True
11463,219060,Male,29,29,young,Caucasian,63,95,Chest/multiple trauma,Onco,170.2,77.1,-45,1724,1724,Alive,Alive,True
12022,222365,Male,45,45,mature,Caucasian,68,103,Chest/extremity trauma,Onco,175.3,NaN,-7,4064,2466,Alive,Alive,True


## 7. Diagnosis Events

In [8]:
diagnosis_cohort = diagnosis[diagnosis["patientunitstayid"].isin(asplenic_ids)].copy()

diagnosis_events = pd.DataFrame({
    "patientunitstayid": diagnosis_cohort["patientunitstayid"],
    "offset": diagnosis_cohort["diagnosisoffset"] if "diagnosisoffset" in diagnosis_cohort.columns else None,
    "type": "diagnosis",
    "event": diagnosis_cohort["diagnosisstring"],
    "code": diagnosis_cohort["icd9code"] if "icd9code" in diagnosis_cohort.columns else None,
    "icd_category": diagnosis_cohort["icd_category"] if "icd_category" in diagnosis_cohort.columns else None,
    "source_table": "diagnosis"
})
diagnosis_events.head()

,patientunitstayid,offset,type,event,code,icd_category,source_table
10415,156843,16968,diagnosis,cardiovascular|arrhythmias|bradycardia,NaN,None,diagnosis
10416,156843,5737,diagnosis,infectious diseases|systemic/other infections|bacteremia|gram negative rod,"038.9, R78.81",None,diagnosis
10417,156843,251,diagnosis,hematology|bleeding and red blood cell disorders|hemorrhage,NaN,None,diagnosis
10418,156843,17365,diagnosis,hematology|platelet disorders|thrombocytopenia|ITP,"287.31, D69.3",None,diagnosis
10419,156843,17365,diagnosis,infectious diseases|systemic/other infections|bacteremia|gram negative rod,"038.9, R78.81",None,diagnosis


## 8. Infection Events

In [9]:
infection_events = diagnosis_events[diagnosis_events["event"].apply(lambda x: match_patterns(x, infection_patterns))].copy()
infection_events["type"] = "infection"
infection_events.head()

,patientunitstayid,offset,type,event,code,icd_category,source_table
10416,156843,5737,infection,infectious diseases|systemic/other infections|bacteremia|gram negative rod,"038.9, R78.81",None,diagnosis
10419,156843,17365,infection,infectious diseases|systemic/other infections|bacteremia|gram negative rod,"038.9, R78.81",None,diagnosis
10425,156843,49040,infection,infectious diseases|chest/pulmonary infections|pneumonia|community-acquired|MTB,"011.60, A15.0",None,diagnosis
10426,156843,17365,infection,cardiovascular|shock / hypotension|septic shock,"785.52, R65.21",None,diagnosis
10429,156843,251,infection,cardiovascular|shock / hypotension|sepsis,"038.9, A41.9",None,diagnosis


## 9. Procedure and Treatment Events

In [10]:
treatment_cohort = treatment[treatment["patientunitstayid"].isin(asplenic_ids)].copy()
treatment_offset_col = "treatmentoffset" if "treatmentoffset" in treatment_cohort.columns else None

procedure_events = pd.DataFrame({
    "patientunitstayid": treatment_cohort["patientunitstayid"],
    "offset": treatment_cohort[treatment_offset_col] if treatment_offset_col else None,
    "type": "procedure",
    "event": treatment_cohort["treatmentstring"],
    "code": None,
    "icd_category": None,
    "source_table": "treatment"
})
procedure_events.head()

,patientunitstayid,offset,type,event,code,icd_category,source_table
7901,247722,2026,procedure,pulmonary|vascular disorders|VTE prophylaxis,None,None,treatment
7902,247722,750,procedure,pulmonary|vascular disorders|VTE prophylaxis|compression boots,None,None,treatment
7903,247722,2026,procedure,gastrointestinal|medications|stress ulcer prophylaxis,None,None,treatment
7904,247722,53,procedure,gastrointestinal|medications|stress ulcer prophylaxis|famotidine,None,None,treatment
7905,247722,750,procedure,gastrointestinal|medications|stress ulcer prophylaxis|famotidine,None,None,treatment


## 10. Splenectomy Events

In [11]:
splenectomy_events = procedure_events[procedure_events["event"].apply(lambda x: match_patterns(x, splenectomy_patterns))].copy()
splenectomy_events["type"] = "splenectomy"
splenectomy_events.head()

,patientunitstayid,offset,type,event,code,icd_category,source_table


## 11. Medication Events

In [12]:
medication_cohort = medication[medication["patientunitstayid"].isin(asplenic_ids)].copy()

offset_col = None
for candidate in ["drugstartoffset", "drugorderoffset", "drugstopoffset"]:
    if candidate in medication_cohort.columns:
        offset_col = candidate
        break

medication_events = pd.DataFrame({
    "patientunitstayid": medication_cohort["patientunitstayid"],
    "offset": medication_cohort[offset_col] if offset_col else None,
    "type": "medication",
    "event": medication_cohort["drugname"],
    "code": None,
    "icd_category": None,
    "source_table": "medication"
})
medication_events.head()

,patientunitstayid,offset,type,event,code,icd_category,source_table
20674,156843,31968,medication,LEVETIRACETAM 500 MG PO TABS,None,None,medication
20675,156843,27153,medication,METHYLPREDNISOLONE SODIUM SUCC 40 MG IJ SOLR,None,None,medication
20676,156843,19115,medication,NaN,None,None,medication
20677,156843,34533,medication,2 ML - METOCLOPRAMIDE HCL 5 MG/ML IJ SOLN,None,None,medication
20678,156843,41726,medication,NaN,None,None,medication


## 12. Vaccination Events

In [13]:
vaccination_events = medication_events[medication_events["event"].apply(lambda x: match_patterns(x, vaccine_patterns))].copy()
vaccination_events["type"] = "vaccination"
vaccination_events.head()

,patientunitstayid,offset,type,event,code,icd_category,source_table
631221,1458348,1858,vaccination,PNEUMOCOCCAL VAC POLYVALENT 25 MCG/0.5ML IJ INJ,None,None,medication
1438365,3168503,676,vaccination,PNEUMOCOCCAL VAC POLYVALENT 25 MCG/0.5ML IJ INJ,None,None,medication
1439834,3169767,33,vaccination,PNEUMOCOCCAL VAC POLYVALENT 25 MCG/0.5ML IJ INJ,None,None,medication
1443044,3173158,268,vaccination,PNEUMOCOCCAL VAC POLYVALENT 25 MCG/0.5ML IJ INJ,None,None,medication
1449577,3179283,61,vaccination,PNEUMOCOCCAL VAC POLYVALENT 25 MCG/0.5ML IJ INJ,None,None,medication


## 13. Admission and Outcome Events

In [14]:
admission_offset = patient_cohort["unitadmitoffset"] if "unitadmitoffset" in patient_cohort.columns else 0
discharge_offset = patient_cohort["unitdischargeoffset"] if "unitdischargeoffset" in patient_cohort.columns else None

admission_events = pd.DataFrame({
    "patientunitstayid": patient_cohort["patientunitstayid"],
    "offset": admission_offset,
    "type": "admission",
    "event": "ICU admission",
    "code": None,
    "icd_category": None,
    "source_table": "patient"
})

discharge_events = pd.DataFrame({
    "patientunitstayid": patient_cohort["patientunitstayid"],
    "offset": discharge_offset,
    "type": "outcome",
    "event": patient_cohort["hospitaldischargestatus"].apply(lambda x: f"Hospital discharge status: {x}"),
    "code": None,
    "icd_category": None,
    "source_table": "patient"
})
admission_events.head()

,patientunitstayid,offset,type,event,code,icd_category,source_table
2348,156843,0,admission,ICU admission,None,None,patient
6191,182613,0,admission,ICU admission,None,None,patient
9381,205175,0,admission,ICU admission,None,None,patient
11463,219060,0,admission,ICU admission,None,None,patient
12022,222365,0,admission,ICU admission,None,None,patient


## 14. Unified Event Timeline with Synthetic Dates

In [15]:
all_events = pd.concat([
    admission_events, diagnosis_events, infection_events, procedure_events,
    splenectomy_events, medication_events, vaccination_events, discharge_events
], ignore_index=True)

all_events["offset"] = pd.to_numeric(all_events["offset"], errors="coerce")
all_events["date"] = all_events["offset"].apply(offset_to_synthetic_datetime)
all_events["date_is_synthetic"] = True
all_events = all_events.dropna(subset=["patientunitstayid"])
all_events = all_events.sort_values(["patientunitstayid", "date", "offset"], na_position="last").reset_index(drop=True)

print("Total events:", len(all_events))
all_events.head(20)

Total events: 63068


,patientunitstayid,offset,type,event,code,icd_category,source_table,date,date_is_synthetic
0,156843,-584,medication,VANCOMYCIN 1.25 GM IN NS 250 ML IVPB (REPACKAGE),None,None,medication,2099-12-31 14:16:00,True
1,156843,-584,medication,CEFEPIME HCL 2 G IJ SOLR,None,None,medication,2099-12-31 14:16:00,True
2,156843,-566,medication,MORPHINE INJ,None,None,medication,2099-12-31 14:34:00,True
3,156843,-533,medication,ACETAMINOPHEN 500 MG PO TABS,None,None,medication,2099-12-31 15:07:00,True
4,156843,-489,medication,30 ML - IOPAMIDOL 61 % IV SOLN,None,None,medication,2099-12-31 15:51:00,True
5,156843,-446,medication,1000 ML - SODIUM CHLORIDE 0.9 % IV SOLN,None,None,medication,2099-12-31 16:34:00,True
6,156843,-420,medication,MORPHINE INJ,None,None,medication,2099-12-31 17:00:00,True
7,156843,-304,medication,BISACODYL 10 MG RE SUPP,None,None,medication,2099-12-31 18:56:00,True
8,156843,-304,medication,MAGNESIUM SULFATE 2 G IN NS PREMIX,None,None,medication,2099-12-31 18:56:00,True
9,156843,-304,medication,100 ML - MAGNESIUM SULFATE IN D5W 10-5 MG/ML-% IV SOLN,None,None,medication,2099-12-31 18:56:00,True


## 15. Optional Deduplication

In [16]:
DEDUPLICATE_EVENTS = False

if DEDUPLICATE_EVENTS:
    all_events = all_events.drop_duplicates(subset=["patientunitstayid", "date", "type", "event", "source_table"]).reset_index(drop=True)

print("Events after optional deduplication:", len(all_events))

Events after optional deduplication: 63068


## 16. Cohort Summary

In [17]:
summary = {
    "n_icu_stays": int(patient_cohort["patientunitstayid"].nunique()),
    "n_events": int(len(all_events)),
    "n_diagnosis_events": int((all_events["type"] == "diagnosis").sum()),
    "n_infection_events": int((all_events["type"] == "infection").sum()),
    "n_procedure_events": int((all_events["type"] == "procedure").sum()),
    "n_splenectomy_events": int((all_events["type"] == "splenectomy").sum()),
    "n_medication_events": int((all_events["type"] == "medication").sum()),
    "n_vaccination_events": int((all_events["type"] == "vaccination").sum()),
    "n_outcome_events": int((all_events["type"] == "outcome").sum()),
    "disease_categories": patient_cohort["primary_disease_category"].value_counts(dropna=False).to_dict()
}
summary

{'n_icu_stays': 444,
 'n_events': 63068,
 'n_diagnosis_events': 23515,
 'n_infection_events': 734,
 'n_procedure_events': 18443,
 'n_splenectomy_events': 0,
 'n_medication_events': 19477,
 'n_vaccination_events': 11,
 'n_outcome_events': 444,
 'disease_categories': {'Onco': 439, 'CHA': 5}}

## 17. JSON Serialization

In [18]:
import numpy as np
patients_json = []

for pid, group in tqdm(all_events.groupby("patientunitstayid"), desc="Building JSON"):
    static_row = patient_static[patient_static["patientunitstayid"] == pid]
    if static_row.empty:
        continue
    p = static_row.iloc[0]
    events = []
    for _, row in group.iterrows():
        event = {
            "date": clean_datetime(row.get("date")),
            "date_is_synthetic": True,
            "offset": clean_value(row.get("offset")),
            "type": clean_value(row.get("type")),
            "event": clean_value(row.get("event")),
            "code": clean_value(row.get("code")),
            "icd_category": clean_value(row.get("icd_category")),
            "source_table": clean_value(row.get("source_table")),
        }
        events.append(event)
    patient_record = {
        "id": int(pid),
        "gender": clean_value(p.get("gender")),
        "age": clean_value(p.get("age")),
        "age_numeric": clean_value(p.get("age_numeric")),
        "age_group": clean_value(p.get("age_group")),
        "ethnicity": clean_value(p.get("ethnicity")),
        "primary_diagnosis": clean_value(p.get("apacheadmissiondx")),
        "primary_disease_category": clean_value(p.get("primary_disease_category")),
        "hospital_discharge_status": clean_value(p.get("hospitaldischargestatus")),
        "unit_discharge_status": clean_value(p.get("unitdischargestatus")),
        #"is_alive?": clean_value(p.get("alive")),
        "is_alive?": "YES" if clean_value(p.get("alive")) else "NO",
        "is_splenectomized?": "NO",
        "date_reference": SYNTHETIC_REFERENCE_DATE.isoformat(),
        "date_reference_is_synthetic": True,
        "events": events
    }
    patients_json.append(patient_record)

events_for_csv = all_events.copy()
events_for_csv["date"] = events_for_csv["date"].apply(clean_datetime)

def json_safe(obj):
    if isinstance(obj, np.bool_):
        return bool(obj)

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()

    if pd.isna(obj):
        return None

    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")
    
with open(OUTPUT_JSON, "w") as f: #, encoding="utf-8") as f:
    json.dump(patients_json, f, indent=2, default=json_safe)#, ensure_ascii=False)

events_for_csv.to_csv(OUTPUT_EVENTS_CSV, index=False)
patient_static.to_csv(OUTPUT_COHORT_CSV, index=False)

print(f"Saved JSON: {OUTPUT_JSON}")
print(f"Saved events CSV: {OUTPUT_EVENTS_CSV}")
print(f"Saved cohort CSV: {OUTPUT_COHORT_CSV}")
print(f"Number of JSON patient records: {len(patients_json)}")

Building JSON:   0%|          | 0/444 [00:00<?, ?it/s]

Saved JSON: eicu_asplenia_patients_synthetic_dates.json
Saved events CSV: eicu_asplenia_events_synthetic_dates.csv
Saved cohort CSV: eicu_asplenia_cohort.csv
Number of JSON patient records: 444


## 18. Inspect One Patient Record

In [19]:
if patients_json:
    print(json.dumps(patients_json[0], indent=2, default=json_safe)[:5000])
else:
    print("No patients found. Check DATA_PATH, ICD_CODE_LIST_DIR, and cohort selection settings.")

{
  "id": 156843,
  "gender": "Female",
  "age": "31",
  "age_numeric": 31,
  "age_group": "young",
  "ethnicity": "Asian",
  "primary_diagnosis": "Sepsis, other",
  "primary_disease_category": "Onco",
  "hospital_discharge_status": "Expired",
  "unit_discharge_status": "Expired",
  "is_alive?": "NO",
  "is_splenectomized?": "NO",
  "date_reference": "2100-01-01T00:00:00",
  "date_reference_is_synthetic": true,
  "events": [
    {
      "date": "2099-12-31T14:16:00",
      "date_is_synthetic": true,
      "offset": -584,
      "type": "medication",
      "event": "VANCOMYCIN 1.25 GM IN NS 250 ML IVPB (REPACKAGE)",
      "code": null,
      "icd_category": null,
      "source_table": "medication"
    },
    {
      "date": "2099-12-31T14:16:00",
      "date_is_synthetic": true,
      "offset": -584,
      "type": "medication",
      "event": "CEFEPIME HCL 2 G IJ SOLR",
      "code": null,
      "icd_category": null,
      "source_table": "medication"
    },
    {
      "date": "2099-12-

## 19. Notes on Synthetic Dates

eICU does not provide absolute calendar dates. The generated `date` field is synthetic:

- offset `0` becomes `2100-01-01T00:00:00`;
- offset `60` becomes `2100-01-01T01:00:00`;
- offset `-1440` becomes `2099-12-31T00:00:00`.

This preserves temporal order and allows algorithms designed for MIMIC-IV date fields to operate on eICU-derived timelines.
